# Atelier complémentaire A : l'effort de R&D en Europe

Cours de M1 Économie de l'innovation, Université de Picardie Jules Verne

**La question de l'atelier.** Comment l'intensité de R&D évolue dans plusieurs économies européennes et ce que l'on peut conclure d'un écart entre pays.

| | |
|---|---|
| Niveau | Pour aller plus loin, à distance, après le TD 2 |
| Ce que l'atelier ajoute aux TD | télécharger les données directement depuis l'API de la Banque mondiale |
| Utile pour le projet | si votre base vient d'une source accessible par API |

Cet atelier est facultatif. Ne pas le faire n'est jamais pénalisé.

### Objectifs

- télécharger une série réelle depuis l'API de la Banque mondiale,
- distinguer le niveau de dépense et l'intensité de R&D,
- contrôler la couverture, les valeurs manquantes, les unités et le dénominateur,
- construire des comparaisons dans le temps et les interpréter à l'aide des chapitres sur les systèmes d'innovation et les externalités,
- expliquer pourquoi une association entre R&D et performance n'établit pas un effet causal.

## Données et provenance

L'indicateur `GB.XPD.RSDV.GD.ZS` mesure les dépenses de recherche et développement en pourcentage du PIB. Les données sont harmonisées et publiées dans les World Development Indicators de la Banque mondiale. La fréquence est annuelle, l'unité d'observation est le pays-année et la période demandée va de 2000 à 2023. Les données peuvent être révisées et ne sont pas disponibles pour chaque pays-année.

La dépense rapportée au PIB décrit une intensité. Elle ne dit ni le montant total ni l'efficacité de la recherche. Elle ne mesure pas toutes les formes d'innovation et n'identifie pas les rendements de la R&D. Source : https://api.worldbank.org/v2/. Métadonnées : https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS

Le notebook utilise `pandas` et `matplotlib`. Les appels au site de la Banque mondiale passent par la bibliothèque standard `urllib`. Il faut une connexion Internet.

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
import matplotlib.pyplot as plt

COUNTRIES = ['FRA', 'DEU', 'SWE', 'FIN', 'ITA', 'ESP', 'NLD', 'POL']
START_YEAR = 2000
END_YEAR = 2023
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Télécharger la série

La fonction lit toutes les pages de résultats de l'API et conserve les valeurs manquantes. Il n'est pas nécessaire de comprendre chaque ligne de la fonction pour l'utiliser.

In [ ]:
def fetch_world_bank_indicator(indicator, start_year=START_YEAR, end_year=END_YEAR):
    base_url = f'https://api.worldbank.org/v2/country/all/indicator/{indicator}'
    rows = []
    page = 1
    total_pages = 1

    while page <= total_pages:
        query = urlencode({
            'format': 'json',
            'date': f'{start_year}:{end_year}',
            'per_page': 1000,
            'page': page,
        })
        with urlopen(f'{base_url}?{query}', timeout=30) as response:
            payload = json.loads(response.read().decode('utf-8'))

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f'Réponse API inattendue pour {indicator}: {payload}')

        metadata, records = payload[0], payload[1]
        total_pages = int(metadata.get('pages') or 1)
        for record in records or []:
            country = record.get('country') or {}
            rows.append({
                'country_code': record.get('countryiso3code'),
                'country': country.get('value'),
                'year': int(record['date']),
                'value': record.get('value'),
                'indicator': indicator,
            })
        page += 1

    return pd.DataFrame(rows)

In [ ]:
R_AND_D_INDICATOR = 'GB.XPD.RSDV.GD.ZS'
try:
    rd_raw = fetch_world_bank_indicator(R_AND_D_INDICATOR)
    rd = rd_raw.loc[rd_raw['country_code'].isin(COUNTRIES)].copy()
    rd['value'] = pd.to_numeric(rd['value'], errors='coerce')
    print(f'{len(rd):,} observations pays-année après sélection.')
except (OSError, ValueError, KeyError) as error:
    rd = pd.DataFrame()
    print("Aucune donnée disponible : vérifiez l'accès Internet ou réessayez.")
    print(f'Détail : {error}')

## 2. Vérifications avant analyse

Examinez la structure et la couverture avant tout calcul. Une valeur manquante signifie que la source ne fournit pas l'observation. Ce n'est pas une dépense nulle.

In [ ]:
if not rd.empty:
    print('Dimensions :', rd.shape)
    print('Années :', rd['year'].min(), 'à', rd['year'].max())
    print('Doublons pays-année :', rd.duplicated(['country_code', 'year']).sum())
    display(rd.groupby('country')['value'].agg(['count', 'min', 'max']))
    display(rd.isna().sum().rename('valeurs manquantes').to_frame())
else:
    print("Aucune donnée disponible : vérifiez l'accès Internet ou réessayez.")

## 3. Comparer les trajectoires

### Tâche A

Produisez un graphique des intensités de R&D dans le temps. Gardez les valeurs manquantes et indiquez les unités. Décrivez ensuite les trajectoires sans classer mécaniquement les systèmes d'innovation.

In [ ]:
if not rd.empty:
    rd_pivot = rd.pivot(index='year', columns='country', values='value')
    ax = rd_pivot.plot(figsize=(11, 6), linewidth=2)
    ax.set(title='Dépenses de R&D en pourcentage du PIB',
           xlabel='Année', ylabel='R&D (% du PIB)')
    ax.legend(title='Pays', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('Figure non produite : la table est vide.')

**À vous.** Repérez un épisode de hausse, de baisse ou de stabilité. Indiquez l'écart entre pays qui vous semble le plus important et vérifiez s'il résiste au choix de la période. Citez une limite des données et un mécanisme du cours qui pourrait expliquer l'écart.

*Votre réponse :*

### Tâche B : niveaux récents et couverture

Comparez les pays pour la dernière année commune disponible et non pour la dernière année observée de chacun. Le tableau ci-dessous calcule cette année commune et indique les valeurs manquantes.

In [ ]:
if not rd.empty:
    coverage = rd.pivot(index='year', columns='country_code', values='value')
    common_years = coverage.dropna().index
    if len(common_years):
        common_year = int(common_years.max())
        latest_common = coverage.loc[common_year].sort_values(ascending=False)
        print('Dernière année commune :', common_year)
        display(latest_common.rename('R&D (% PIB)').to_frame())
    else:
        print('Aucune année commune complète : comparez la couverture pays par pays.')
else:
    print('Aucune donnée disponible.')

## 4. Relier l'indicateur au cours

Les dépenses de R&D sont un intrant. Des externalités peuvent faire que les rendements sociaux dépassent les rendements privés. La façon dont les agents de l'innovation s'organisent influe sur la circulation et l'absorption des connaissances. Une intensité de R&D élevée ne prouve pas à elle seule que les connaissances circulent mieux, que les entreprises innovent davantage ou que le bien-être augmente.

**À vous.** Répondez en quelques lignes à chaque point :

1. l'intérêt de rapporter la R&D au PIB plutôt que de comparer des montants,
2. l'effet possible de la spécialisation sectorielle d'un pays sur ce ratio (cf. Chapitre 5),
3. les données qui permettraient de distinguer l'effort de recherche, l'innovation produite et sa diffusion,
4. deux mécanismes qui produiraient une association entre R&D et productivité sans effet causal direct de la dépense,
5. les canaux d'externalités qui pourraient relier des entreprises ou des pays (cf. Chapitre 10).

## 5. Prolongement : R&D et revenu par habitant

Téléchargez l'indicateur `NY.GDP.PCAP.PP.KD` (PIB par habitant, dollars internationaux constants) avec la même fonction et réunissez les deux séries par pays et par année. Tracez un nuage de points pour une année commune, calculez une corrélation et commentez ses limites. Les pays ne choisissent pas leur niveau de R&D au hasard. Le revenu, les institutions, la structure sectorielle, l'éducation et les infrastructures peuvent agir sur les deux variables. Une corrélation ne mesure donc pas un rendement de la R&D.

## Pour le projet

Les meilleurs projets des années précédentes partaient de données réelles, décrivaient l'unité d'observation, interprétaient les résultats à l'aide du cours et reconnaissaient leurs limites.

Pour votre projet, formulez une question, citez vos sources, contrôlez les données, choisissez vos variables, produisez des statistiques descriptives et des graphiques, reliez chaque résultat à un mécanisme et discutez les limites. Si vous ajoutez une régression (cf. TD 4), expliquez le coefficient et ce qu'il ne prouve pas. Un résultat prudent et bien interprété vaut mieux qu'une causalité affirmée sans justification.

## Limites et vérifications

- [ ] La variable, l'unité, la période et le dénominateur sont explicités.
- [ ] Les doublons et les valeurs manquantes sont contrôlés sans être remplacés arbitrairement.
- [ ] Les graphiques ont un titre, des axes, des unités et une source.
- [ ] Je distingue dépense, innovation, diffusion et performance.
- [ ] Les comparaisons tiennent compte des différences de structure entre pays.
- [ ] Aucune causalité n'est déduite d'une simple comparaison.
- [ ] La date de téléchargement et les références sont indiquées.

### Références

- Banque mondiale, World Development Indicators : https://data.worldbank.org/
- Griliches, Z. (1992), The Search for R&D Spillovers, *Scandinavian Journal of Economics*, 94, S29-S47.
- OCDE (2015), *Frascati Manual 2015*.
- OCDE/Eurostat (2018), *Oslo Manual 2018*.